# PCBUILD QLoRA v20: direct-answer refinement từ adapter v18

V18 có validation loss 0.8902; v19 dừng sớm và không cải thiện validation. Lượt v20 đầu khởi chạy với micro-batch 2 nhưng CUDA OOM ngay tại 0/90, khi VRAM trống dưới 800 MiB; lượt đó không tạo checkpoint hay kết quả đánh giá. Cell setup dưới đây dừng trước khi cài gói nếu không có GPU hoặc còn dưới 10 GiB VRAM trống.

Bản retry dùng adapter tốt nhất v18 làm điểm khởi đầu, micro-batch 1 × accumulation 4 (effective batch 4), learning rate 2e-6, tối đa 2 epochs và early stopping. Dữ liệu có thêm chỉ thị trả lời trực tiếp; suffix có thể làm thay đổi số token còn nhãn sau truncation, vì vậy validation loss v20 chỉ là chỉ báo và không được coi là so sánh chắc chắn cùng split. Các ca behavior held-out được giữ ngoài dữ liệu train và cần chạy riêng sau khi có adapter hoàn tất.

V18/v19 và log v20 lỗi được giữ trên Drive. Bản retry có `RUN_NAME` riêng để không resume nhầm cấu hình batch 2. QLoRA adapter vẫn là artifact offline; app hiện không tự nạp adapter này.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import hashlib
import json
import shutil
import subprocess
import sys
import time
import zipfile
from pathlib import Path
from google.colab import drive, files

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/PCBUILD")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
BUNDLE_PATH = DRIVE_ROOT / "pcbuild-colab-bundle.zip"

LOCAL_BUNDLE = Path("/content/pcbuild-colab-bundle-v15.zip")
if not BUNDLE_PATH.exists() or hashlib.sha256(BUNDLE_PATH.read_bytes()).hexdigest() != "17ce86994806073558a7156b2986b4164a040676158c3a8cfbabe97c31bb6958":
    if not LOCAL_BUNDLE.is_file():
        raise FileNotFoundError(f"Upload the new bundle with the VS Code Colab Explorer: {LOCAL_BUNDLE}")
    bundle_bytes = LOCAL_BUNDLE.read_bytes()
    if hashlib.sha256(bundle_bytes).hexdigest() != "17ce86994806073558a7156b2986b4164a040676158c3a8cfbabe97c31bb6958":
        raise ValueError("Uploaded Colab bundle does not match the expected SHA-256.")
    BUNDLE_PATH.write_bytes(bundle_bytes)
    print("Verified the v15 bundle and copied it to Google Drive.")

with zipfile.ZipFile(BUNDLE_PATH) as archive:
    manifest = json.loads(archive.read("COLAB_BUNDLE_MANIFEST.json"))
    for name, expected_hash in manifest["files"].items():
        content = archive.read(name)
        if hashlib.sha256(content).hexdigest() != expected_hash:
            raise ValueError(f"Bundle checksum mismatch: {name}")
    for member in archive.namelist():
        path = Path(member)
        if path.is_absolute() or ".." in path.parts:
            raise ValueError(f"Unsafe archive path: {member}")
    PROJECT = Path("/content/pcbuild")
    if PROJECT.exists():
        shutil.rmtree(PROJECT)
    archive.extractall(PROJECT, members=[item for item in archive.infolist() if item.filename != "COLAB_BUNDLE_MANIFEST.json"])

TRAINING = PROJECT / "training"
print(f"Verified {len(manifest['files'])} files; extracted source to {PROJECT}")
print(f"Persistent Drive directory: {DRIVE_ROOT}")

## Cài môi trường và xác nhận GPU

Colab cần bật GPU tại Runtime → Change runtime type. Cell này giữ nguyên PyTorch/CUDA mà Colab cung cấp, chỉ cài các thư viện fine-tune đã được pin. Nếu GPU không hỗ trợ bfloat16, script tự chuyển sang float16.

In [ ]:
import subprocess
import sys
import torch

# Fail before dependency installation or model loading if Colab did not assign a usable GPU.
if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU. Enable a GPU runtime in Google Colab and rerun.")

free_bytes, total_bytes = torch.cuda.mem_get_info()
free_gib = free_bytes / (1024 ** 3)
total_gib = total_bytes / (1024 ** 3)
MINIMUM_FREE_GPU_GIB = 10.0
print(f"GPU: {torch.cuda.get_device_name(0)}; CUDA: {torch.version.cuda}")
print(f"GPU memory free: {free_gib:.2f}/{total_gib:.2f} GiB")
if free_gib < MINIMUM_FREE_GPU_GIB:
    raise RuntimeError(
        f"Only {free_gib:.2f} GiB GPU memory is free; need at least "
        f"{MINIMUM_FREE_GPU_GIB:.1f} GiB before installing packages or training. "
        "Reconnect to a fresh GPU runtime and rerun all cells."
    )

packages = [
    "transformers==5.20.2",
    "peft==0.21.2",
    "accelerate==1.15.0",
    "safetensors==0.8.0",
    "bitsandbytes==0.50.2",
    "Pillow==12.3.0",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])

# Repair mixed Pillow files left by interrupted or overlapping Colab installs.
pillow_pin = next(package for package in packages if package.lower().startswith("pillow=="))
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir",
    "--force-reinstall", "--no-deps", pillow_pin,
])
pillow_check = subprocess.run(
    [
        sys.executable,
        "-c",
        "import PIL; import PIL._typing; import PIL.ImageText; print(PIL.__version__)",
    ],
    check=True,
    capture_output=True,
    text=True,
)
print("Fresh-process Pillow import check:", pillow_check.stdout.strip())

# Colab may have imported an older Pillow before this repair; reload it in this kernel.
for module_name in tuple(sys.modules):
    if module_name == "PIL" or module_name.startswith("PIL."):
        del sys.modules[module_name]
import PIL
from PIL import ImageText as pillow_image_text
print("Kernel Pillow import check:", PIL.__version__, pillow_image_text.__name__)
capability = torch.cuda.get_device_capability(0)
print("native bf16:", torch.cuda.is_bf16_supported() and capability[0] >= 8)
print("HF auth token: not required for the public base model")

## Dataset và cấu hình v20

Kết hợp 203 ví dụ general với 52 ví dụ corrective đã rà soát; script kiểm tra schema và prompt trùng. V20 thêm một chỉ thị trả lời trực tiếp tổng quát vào system prompt. Vì tokenizer cắt chuỗi ở giới hạn 768 token và chỉ giữ ví dụ còn token assistant để học, hậu tố có thể làm thay đổi tập encoded và membership của validation. Notebook in số lượng ví dụ sau encode để phát hiện chênh lệch; số lượng trùng nhau vẫn không chứng minh validation membership giống hệt.

Run retry khởi tạo từ adapter v18, seed 42, micro-batch 1 × accumulation 4 (effective batch 4), learning rate 2e-6, tối đa 2 epochs. Run dùng thư mục Drive mới, tách biệt log/checkpoint lỗi trước đó.

In [ ]:
subprocess.check_call([
    sys.executable,
    str(TRAINING / "prepare_colab_dataset.py"),
    "--output",
    str(TRAINING / "data" / "pc_colab_sft.jsonl"),
])
SOURCE_DATA_FILE = TRAINING / "data" / "pc_colab_sft.jsonl"
DATA_FILE = TRAINING / "data" / "pc_colab_sft_v20.jsonl"
V20_SYSTEM_SUFFIX = "\nKhông hiển thị quá trình suy nghĩ nội bộ; chỉ trả lời trực tiếp bằng tiếng Việt, nêu kết luận và dữ kiện cần thiết."
with SOURCE_DATA_FILE.open(encoding="utf-8") as source, DATA_FILE.open("w", encoding="utf-8", newline="\n") as target:
    for line in source:
        row = json.loads(line)
        row["messages"][0]["content"] += V20_SYSTEM_SUFFIX
        target.write(json.dumps(row, ensure_ascii=False) + "\n")
BASELINE_RUN_NAME = "pcbuild-qwen3.5-2b-qlora-v14-colab"
BASELINE_ADAPTER = DRIVE_ROOT / "runs" / BASELINE_RUN_NAME / "checkpoints" / "final"
PARENT_RUN_NAME = "pcbuild-qwen3.5-4b-qlora-v18-mb2-colab"
PARENT_RUN_DRIVE = DRIVE_ROOT / "runs" / PARENT_RUN_NAME
SEED_ADAPTER = PARENT_RUN_DRIVE / "checkpoints" / "final"
RUN_NAME = "pcbuild-qwen3.5-4b-qlora-v20-direct-answer-mb1-colab"
MODEL_ID = "Qwen/Qwen3.5-4B"
RUN_DRIVE = DRIVE_ROOT / "runs" / RUN_NAME
RUN_DRIVE.mkdir(parents=True, exist_ok=True)
if not (SEED_ADAPTER / "adapter_model.safetensors").is_file():
    raise FileNotFoundError(f"Parent v18 adapter missing: {SEED_ADAPTER}")
if not (BASELINE_ADAPTER / "training_metrics.json").is_file():
    raise FileNotFoundError(f"v14 comparison metrics missing: {BASELINE_ADAPTER}")

# The packaged Colab train script predates Qwen direct-answer mode; patch both prompt and target templates.
training_script = TRAINING / "train_qlora.py"
training_source = training_script.read_text(encoding="utf-8")
template_calls = [
    'messages[:-1], tokenize=True, add_generation_prompt=True,\n            return_dict=True, return_tensors="pt"',
    'messages, tokenize=True, add_generation_prompt=False,\n            return_dict=True, return_tensors="pt"',
]
for template_call in template_calls:
    target_call = template_call.replace(
        'return_dict=True, return_tensors="pt"',
        'return_dict=True, return_tensors="pt", enable_thinking=False',
    )
    if target_call not in training_source:
        if template_call not in training_source:
            raise ValueError("Expected train chat-template call missing; refusing an unverified patch.")
        training_source = training_source.replace(template_call, target_call, 1)
training_script.write_text(training_source, encoding="utf-8")



def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def sha256_tree(root):
    digest = hashlib.sha256()
    for path in sorted(item for item in root.rglob("*") if item.is_file()):
        digest.update(path.relative_to(root).as_posix().encode("utf-8"))
        digest.update(bytes.fromhex(sha256_file(path)))
    return digest.hexdigest()

PARENT_METRICS = json.loads((SEED_ADAPTER / "training_metrics.json").read_text(encoding="utf-8"))
BASELINE_METRICS = json.loads((BASELINE_ADAPTER / "training_metrics.json").read_text(encoding="utf-8"))
RUN_CONFIG = {
    "baseline_run": BASELINE_RUN_NAME,
    "baseline_validation_loss": BASELINE_METRICS["eval_loss"],
    "parent_run": PARENT_RUN_NAME,
    "parent_validation_loss": PARENT_METRICS["eval_loss"],
    "model": MODEL_ID,
    "init_adapter": str(SEED_ADAPTER),
    "seed_adapter_sha256": sha256_tree(SEED_ADAPTER),
    "dataset_sha256": sha256_file(DATA_FILE),
    "training_script_sha256": sha256_file(TRAINING / "train_qlora.py"),
    "dataset_builder_sha256": sha256_file(TRAINING / "prepare_colab_dataset.py"),
    "dataset_rows": sum(1 for line in DATA_FILE.open(encoding="utf-8") if line.strip()),
    "epochs": 2,
    "learning_rate": 2e-6,
    "micro_batch_size": 1,
    "effective_batch_size": 4,
    "minimum_free_gpu_gib": MINIMUM_FREE_GPU_GIB,
    "gradient_accumulation_steps": 4,
    "max_length": 768,
    "seed": 42,
    "save_eval_steps": 8,
    "early_stopping_patience": 3,
    "thinking_mode": "disabled",
    "system_prompt_suffix": V20_SYSTEM_SUFFIX,
}
manifest_path = RUN_DRIVE / "run_config.json"
if manifest_path.exists():
    existing = json.loads(manifest_path.read_text(encoding="utf-8"))
    if existing != RUN_CONFIG:
        raise ValueError("Run config/data changed. Use a new RUN_NAME to avoid resuming an incompatible checkpoint.")
else:
    manifest_path.write_text(json.dumps(RUN_CONFIG, indent=2), encoding="utf-8")
print(json.dumps(RUN_CONFIG, indent=2))

## Validation seed adapter v14

Đọc metrics đã lưu của adapter v14 và adapter cha v18 để có mốc tham khảo trước khi chạy retry. So sánh loss chỉ có ý nghĩa khi tập encoded và membership validation tương ứng; cell cuối sẽ in số lượng ví dụ để kiểm tra sơ bộ. Behavioral held-out evaluation là phép đánh giá riêng.

In [ ]:
PARENT_METRICS = json.loads((SEED_ADAPTER / "training_metrics.json").read_text(encoding="utf-8"))
print(f"v14 validation loss: {BASELINE_METRICS['eval_loss']}")
print(f"v18 parent validation loss: {PARENT_METRICS['eval_loss']}")

## Train v20: direct-answer refinement

Run mới bắt đầu từ adapter tốt nhất v18; train encoder và evaluator đều tắt thinking. Trước pip install, notebook xác nhận GPU và yêu cầu ít nhất 10 GiB VRAM còn trống. Batch 1 × accumulation 4 giữ effective batch 4 để giảm peak VRAM; checkpoint v20 lưu trong thư mục Drive riêng. Lượt v20 batch 2 trước đó OOM tại bước 0/90 và không có checkpoint để resume.

In [ ]:
LOCAL_OUTPUT = TRAINING / "output" / RUN_NAME
CHECKPOINT_DRIVE = RUN_DRIVE / "checkpoints"
LOCAL_OUTPUT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DRIVE.mkdir(parents=True, exist_ok=True)
FINAL_ADAPTER = CHECKPOINT_DRIVE / "final"
FINAL_METRICS = FINAL_ADAPTER / "training_metrics.json"

if FINAL_METRICS.is_file():
    final_metrics = json.loads(FINAL_METRICS.read_text(encoding="utf-8"))
    print("Run already completed; keeping the saved best adapter.")
    print("Best validation loss:", final_metrics.get("eval_loss"))
    print("Adapter:", FINAL_ADAPTER)
else:
    resume_path = None
    latest_file = CHECKPOINT_DRIVE / "latest_checkpoint.json"
    if latest_file.exists():
        latest = json.loads(latest_file.read_text(encoding="utf-8"))
        checkpoint_name = latest["checkpoint"]
        if Path(checkpoint_name).name != checkpoint_name or not checkpoint_name.startswith("checkpoint-"):
            raise ValueError(f"Unsafe checkpoint name in Drive manifest: {checkpoint_name}")
        drive_checkpoint = CHECKPOINT_DRIVE / checkpoint_name
        state_file = drive_checkpoint / "trainer_state.json"
        if not state_file.is_file():
            raise FileNotFoundError(f"Checkpoint is incomplete: {state_file}")
        state = json.loads(state_file.read_text(encoding="utf-8"))
        if int(state["global_step"]) != int(latest["global_step"]):
            raise ValueError("Drive checkpoint state does not match latest_checkpoint.json")
        local_checkpoint = LOCAL_OUTPUT / checkpoint_name
        if local_checkpoint.exists():
            shutil.rmtree(local_checkpoint)
        shutil.copytree(drive_checkpoint, local_checkpoint)
        resume_path = local_checkpoint
        print(f"Resuming from Drive checkpoint {checkpoint_name} at step {latest['global_step']}")
    else:
        print("No complete checkpoint; continuing from the best v18 adapter with micro-batch 1 / accumulation 4.")

    command = [
        sys.executable, str(TRAINING / "train_qlora.py"),
        "--epochs", str(RUN_CONFIG["epochs"]),
        "--learning-rate", str(RUN_CONFIG["learning_rate"]),
        "--model", RUN_CONFIG["model"],
        "--data", str(DATA_FILE),
        "--output", str(LOCAL_OUTPUT),
        "--save-steps", str(RUN_CONFIG["save_eval_steps"]),
        "--eval-steps", str(RUN_CONFIG["save_eval_steps"]),
        "--logging-steps", "1",
        "--micro-batch-size", str(RUN_CONFIG["micro_batch_size"]),
        "--effective-batch-size", str(RUN_CONFIG["effective_batch_size"]),
        "--early-stopping-patience", str(RUN_CONFIG["early_stopping_patience"]),
        "--checkpoint-backup-dir", str(CHECKPOINT_DRIVE),
    ]
    if RUN_CONFIG["init_adapter"]:
        command.extend(["--init-adapter", str(SEED_ADAPTER)])
    if resume_path:
        command.extend(["--resume-from-checkpoint", str(resume_path)])

    training_log = RUN_DRIVE / "training.log"
    env = dict(**__import__("os").environ)
    env["WANDB_DISABLED"] = "true"
    started_at = time.time()
    with training_log.open("a", encoding="utf-8") as log_stream:
        process = subprocess.Popen(command, cwd=PROJECT, stdout=log_stream, stderr=subprocess.STDOUT, env=env)
        while process.poll() is None:
            try:
                process.wait(timeout=45)
            except subprocess.TimeoutExpired:
                heartbeat_path = CHECKPOINT_DRIVE / "heartbeat.json"
                if heartbeat_path.exists() and heartbeat_path.stat().st_mtime >= started_at:
                    heartbeat = json.loads(heartbeat_path.read_text(encoding="utf-8"))
                    print("Progress:", json.dumps(heartbeat, ensure_ascii=False))
                else:
                    print("Training is loading the model; log:", training_log)

    exit_code = process.returncode
    tail = training_log.read_text(encoding="utf-8", errors="replace").splitlines()[-60:]
    print("\n".join(tail))
    if exit_code:
        raise RuntimeError(f"Training stopped with exit code {exit_code}. Rerun this cell to resume from the latest saved checkpoint.")
    if not FINAL_METRICS.is_file():
        raise FileNotFoundError(f"Training ended without final metrics: {FINAL_METRICS}")
    final_metrics = json.loads(FINAL_METRICS.read_text(encoding="utf-8"))
    print("Best validation loss:", final_metrics.get("eval_loss"))
    print("Final adapter and manifest:", FINAL_ADAPTER)

## So sánh v20 với v18 và v14

So sánh best validation loss cùng số lượng ví dụ đã encode với các metrics đã lưu của v18/v14. Do system suffix có thể làm thay đổi ví dụ bị loại sau truncation, đây không phải bằng chứng chắc chắn rằng validation split giống hệt. Chỉ chọn checkpoint sau khi xem log, số lượng và kết quả; chạy riêng các behavior held-out trước mọi quyết định dùng adapter. Notebook không tự đổi model runtime.

In [ ]:
FINAL_ADAPTER = CHECKPOINT_DRIVE / "final"
V20_METRICS = json.loads((FINAL_ADAPTER / "training_metrics.json").read_text(encoding="utf-8"))
selected_loss = V20_METRICS["eval_loss"]
parent_examples = PARENT_METRICS.get("examples")
v20_examples = V20_METRICS.get("examples")
parent_data_hash = PARENT_METRICS.get("data_sha256")
v20_data_hash = V20_METRICS.get("data_sha256")
print(f"v14 2B validation loss reference: {RUN_CONFIG['baseline_validation_loss']:.4f}")
print(f"v18 parent 4B validation loss reference: {RUN_CONFIG['parent_validation_loss']:.4f}")
print(f"v20 4B best validation loss: {selected_loss:.4f}")
print("Encoded examples: v18=", parent_examples, "; v20=", v20_examples)
counts_available = parent_examples is not None and v20_examples is not None
print("Encoded counts match:", counts_available and parent_examples == v20_examples)
print("Data fingerprints match:", parent_data_hash is not None and parent_data_hash == v20_data_hash)
print("Lower numeric loss than v18 reference:", selected_loss < RUN_CONFIG["parent_validation_loss"])
print("Adapter:", FINAL_ADAPTER)
print("Loss is not a strict same-split comparison unless validation membership is confirmed; run held-out behavior evaluation separately.")

In [ ]:
import importlib.util
import sys
adapter_config = json.loads((FINAL_ADAPTER / "adapter_config.json").read_text(encoding="utf-8"))
eval_base_model = str(adapter_config.get("base_model_name_or_path") or "")
if eval_base_model != RUN_CONFIG["model"]:
    raise ValueError(f"Adapter base mismatch: {eval_base_model!r} != {RUN_CONFIG['model']!r}")
eval_script = TRAINING / "evaluate_adapter.py"
# Adapt the Colab bundle copy for notebook stdout and Qwen direct-answer evaluation.
eval_source = eval_script.read_text(encoding="utf-8")
stdout_call = '    sys.stdout.reconfigure(encoding="utf-8")'
if stdout_call in eval_source:
    stdout_guard = "\n".join([
        '    reconfigure_stdout = getattr(sys.stdout, "reconfigure", None)',
        '    if reconfigure_stdout is not None:',
        '        reconfigure_stdout(encoding="utf-8")',
    ])
    eval_source = eval_source.replace(stdout_call, stdout_guard, 1)
thinking_args = 'return_dict=True, return_tensors="pt"'
if "enable_thinking=False" not in eval_source and thinking_args in eval_source:
    eval_source = eval_source.replace(
        thinking_args,
        thinking_args + ", enable_thinking=False",
        1,
    )
eval_script.write_text(eval_source, encoding="utf-8")
spec = importlib.util.spec_from_file_location("pcbuild_evaluate_adapter_v20", eval_script)
evaluator = importlib.util.module_from_spec(spec)
spec.loader.exec_module(evaluator)
evaluator.BASE = eval_base_model
eval_report = RUN_DRIVE / "qualitative_eval.json"
previous_argv = sys.argv
try:
    sys.argv = [str(eval_script), "--adapter", str(FINAL_ADAPTER), "--output", str(eval_report)]
    evaluator.main()
finally:
    sys.argv = previous_argv